Experiment 2 -- External validity on the real German Credit benchmark

In [1]:
# 02_benchmark_german_credit.ipynb
#
# Experiment 2 -- External validity on the real German Credit benchmark
#
# Purpose:
#   Replicate the core finding on a real public dataset (Statlog German Credit),
#   with the protected attribute = sex (male vs female), across 30 random seeds,
#   reporting 95% confidence intervals. Also report equalized-odds gaps.
#
# Conditions: with_protected, drop_protected, drop_proxies, gov_engine
# Outputs:
#   results/tables/exp2_benchmark_metrics.csv
#   results/figures/exp2_benchmark_dir.(png|pdf)

# ==== Imports and grayscale academic style (600 dpi, PNG+PDF, no captions) ====
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.isdir(os.path.join(PROJECT_ROOT, "results")):
    PROJECT_ROOT = os.getcwd()
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
FIG_DIR = os.path.join(PROJECT_ROOT, "results", "figures")
TAB_DIR = os.path.join(PROJECT_ROOT, "results", "tables")
for d in (DATA_DIR, FIG_DIR, TAB_DIR):
    os.makedirs(d, exist_ok=True)

sns.set_theme(style="whitegrid")
GRAYS = ["#000000", "#555555", "#999999", "#cccccc"]
sns.set_palette(sns.color_palette(GRAYS))
plt.rcParams.update({
    "figure.dpi": 600, "savefig.dpi": 600, "font.size": 11,
    "axes.edgecolor": "black", "axes.linewidth": 0.8, "grid.color": "0.85",
})

def save_fig(fig, name):
    fig.savefig(os.path.join(FIG_DIR, name + ".png"), dpi=600, bbox_inches="tight")
    fig.savefig(os.path.join(FIG_DIR, name + ".pdf"), dpi=600, bbox_inches="tight")

def disparate_impact_ratio(y_pred, group):
    approve = (y_pred == 0).astype(int)
    r1 = approve[group == 1].mean(); r0 = approve[group == 0].mean()
    return min(r1, r0) / max(r1, r0) if max(r1, r0) > 0 else np.nan

def equalized_odds_gaps(y_true, y_pred, group):
    def rate(ct, mask):
        idx = (y_true == ct) & mask
        return (y_pred[idx] == 1).mean() if idx.sum() else np.nan
    a, b = group == 1, group == 0
    return abs(rate(1, a) - rate(1, b)), abs(rate(0, a) - rate(0, b))



from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

df = pd.read_csv(os.path.join(DATA_DIR, "german_credit.csv"))
df["sex_male"] = df["personal_status_sex"].str.startswith("male").astype(int)
df["default"] = (df["credit_risk"] == 0).astype(int)

cat = [c for c in df.select_dtypes("object").columns if c != "personal_status_sex"]
X = pd.get_dummies(df.drop(columns=["credit_risk", "default", "personal_status_sex"]), columns=cat).astype(float)

# Identify the four strongest proxies for the protected attribute
corr = X.drop(columns=["sex_male"]).apply(
    lambda c: abs(np.corrcoef(c, df["sex_male"])[0, 1]) if c.std() > 0 else 0.0)
proxies = corr.sort_values(ascending=False).head(4).index.tolist()

feat_all = X.columns.tolist()
feat_dropP = [c for c in feat_all if c != "sex_male"]
feat_dropPX = [c for c in feat_dropP if c not in proxies]

def run(feats, seed, gov=False):
    Xtr, Xte, ytr, yte, gtr, gte = train_test_split(
        X[feats], df["default"], df["sex_male"], test_size=0.3,
        random_state=seed, stratify=df["default"])
    sc = StandardScaler().fit(Xtr)
    m = LogisticRegression(max_iter=2000).fit(sc.transform(Xtr), ytr)
    p = m.predict_proba(sc.transform(Xte))[:, 1]
    auc = roc_auc_score(yte, p)
    g = gte.values
    if not gov:
        yp = (p >= 0.5).astype(int)
    else:
        base = ((p >= 0.5).astype(int) == 0).mean()
        t1 = np.quantile(p[g == 1], base); t0 = np.quantile(p[g == 0], base)
        yp = np.where(g == 1, (p >= t1), (p >= t0)).astype(int)
    dtpr, dfpr = equalized_odds_gaps(yte.values, yp, g)
    return auc, disparate_impact_ratio(yp, g), dtpr, dfpr

SEEDS = list(range(30))
conditions = [("with_protected", feat_all, False),
              ("drop_protected", feat_dropP, False),
              ("drop_proxies", feat_dropPX, False),
              ("gov_engine", feat_dropP, True)]

rows = []
for name, feats, gov in conditions:
    arr = np.array([run(feats, s, gov) for s in SEEDS])
    mean = arr.mean(0); ci = 1.96 * arr.std(0) / np.sqrt(len(SEEDS))
    rows.append({"condition": name,
                 "AUC": mean[0], "AUC_ci": ci[0],
                 "DIR": mean[1], "DIR_ci": ci[1],
                 "dTPR": mean[2], "dFPR": mean[3]})

res = pd.DataFrame(rows)
res.to_csv(os.path.join(TAB_DIR, "exp2_benchmark_metrics.csv"), index=False)
print("Protected attribute: sex (male vs female); proxies:", proxies)
print(res.round(3).to_string(index=False))

order = ["with_protected", "drop_protected", "drop_proxies", "gov_engine"]
pl = res.set_index("condition").loc[order].reset_index()
fig, ax = plt.subplots(figsize=(6, 4.2))
bars = ax.bar(range(len(pl)), pl["DIR"], yerr=pl["DIR_ci"], capsize=4,
              edgecolor="black", linewidth=1.0,
              color=["#000000", "#555555", "#999999", "#cccccc"])
ax.axhline(0.8, color="black", linestyle="--", linewidth=0.9)
ax.set_xticks(range(len(pl))); ax.set_xticklabels(pl["condition"], rotation=20, ha="right")
ax.set_ylabel("Disparate Impact Ratio (DIR)"); ax.set_ylim(0, 1.05)
fig.tight_layout(); save_fig(fig, "exp2_benchmark_dir"); plt.close(fig)
print("Saved:", os.path.join(TAB_DIR, "exp2_benchmark_metrics.csv"))


Protected attribute: sex (male vs female); proxies: ['housing_rent', 'people_liable', 'employment_duration_... < 1 year', 'age']
     condition   AUC  AUC_ci   DIR  DIR_ci  dTPR  dFPR
with_protected 0.777   0.009 0.844   0.025 0.195 0.066
drop_protected 0.777   0.009 0.920   0.020 0.111 0.045
  drop_proxies 0.778   0.009 0.928   0.018 0.102 0.043
    gov_engine 0.777   0.009 0.997   0.001 0.077 0.035


Saved: /home/claude/project/results/tables/exp2_benchmark_metrics.csv
